# `raster_io` — GeoTIFF Reading Helpers

Thin `rasterio` wrappers for reading Sentinel-2, Sentinel-1, and ground-truth GeoTIFFs into numpy arrays. Converted from `src/data/raster_io.py` into a notebook so it can be opened, read, and run on its own. Other notebooks pull these functions in via the small `load_notebook()` loader defined in their own first cell (see any training notebook).

In [1]:
import rasterio
import numpy as np

def read_s2(path: str) -> np.ndarray:
    """
    Read Sentinel-2 GeoTIFF file.
    Returns:
        np.ndarray of shape (11, H, W) normalized to [0, 1.0] reflectance scale.
    """
    with rasterio.open(path) as src:
        data = src.read().astype(np.float32)
        # Sentinel-2 BOA reflectance values are scaled by 10000.
        data /= 10000.0
        return data

def read_s1(path: str) -> np.ndarray:
    """
    Read Sentinel-1 GeoTIFF file.
    Returns:
        np.ndarray of shape (3, H, W) containing VV, VH, and their ratio, in dB.
    """
    with rasterio.open(path) as src:
        data = src.read().astype(np.float32)
        return data

def read_gt(path: str) -> np.ndarray:
    """
    Read Ground Truth GeoTIFF file.
    Returns:
        np.ndarray of shape (H, W) containing class labels.
    """
    with rasterio.open(path) as src:
        return src.read(1)

print("raster_io loaded: read_s2, read_s1, read_gt")


raster_io loaded: read_s2, read_s1, read_gt


## Demo — read one real training scene

In [2]:
import glob, os, yaml

REPO = r"C:\Users\prajw\OneDrive - Amrita Vishwa Vidyapeetham\Desktop\SEM5\Deep Learning\flood-segmentation-sentinel12"
with open(os.path.join(REPO, "configs", "data.yaml")) as f:
    _cfg = yaml.safe_load(f)

_sample_s2 = sorted(glob.glob(os.path.join(_cfg["dataset_root"], "tifs", "Train", "S2", "*.tif")))[0]
_sample_s1 = _sample_s2.replace(os.sep + "S2" + os.sep, os.sep + "S1" + os.sep)
_sample_gt = _sample_s2.replace(os.sep + "S2" + os.sep, os.sep + "GT" + os.sep)

s2 = read_s2(_sample_s2)
s1 = read_s1(_sample_s1)
gt = read_gt(_sample_gt)

print(f"Sample scene: {os.path.basename(_sample_s2)}")
print(f"S2 shape: {s2.shape}, range [{s2.min():.3f}, {s2.max():.3f}]")
print(f"S1 shape: {s1.shape}, range [{s1.min():.2f}, {s1.max():.2f}]")
print(f"GT shape: {gt.shape}, unique classes: {sorted(np.unique(gt).tolist())}")


Sample scene: BNPI_20210223.tif
S2 shape: (11, 3600, 5400), range [0.000, 1.742]
S1 shape: (3, 3600, 5400), range [-35.45, 21.87]
GT shape: (3600, 5400), unique classes: [2, 3, 4, 5, 6, 7]
